# CrashVLM: GPU runs on Kaggle (steps D and C)

Full guide: `docs/KAGGLE.md` in the repo. In short:

1. **Settings** (right panel): Accelerator **GPU T4 x2**, Internet **on**.
2. **Add Input** → your private dataset `crashvlm-data`.
3. **Add-ons → Secrets**: `GITHUB_TOKEN`, attached to this notebook.
4. Run cells 1 and 2 by hand (**smoke test**, about 15 min). Both must end with `OK`.
5. **Save Version → Save & Run All (Commit)**. The long run (up to about 11 h) happens in the
   background; you can close the browser.
6. When the version finishes: **Output** tab → download `crashvlm_gpu_results.zip`.

In the committed run, the smoke test is skipped and the long steps run. In an interactive
session it is the other way round.


In [ ]:
# 1. Setup: GPU, data, code, install  (~5 min)
import glob, os, subprocess, sys, time, zipfile
from pathlib import Path

T0 = time.time()
BATCH = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"   # True in "Save & Run All"
print("mode:", "COMMIT (long run)" if BATCH else "INTERACTIVE (smoke test)")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

# --- data: link the dataset files into a writable folder ---
need = ["annotation/clip_index.json", "interim/selection.parquet", "sft/train.jsonl",
        "sft/val.jsonl", "sft/train_f0.25.jsonl", "sft/train_f0.5.jsonl"]
DATA = Path("/kaggle/working/data")
os.environ["CRASHVLM_DATA"] = str(DATA)
os.environ["HF_HOME"] = "/tmp/hf"          # model downloads stay out of the saved output

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return Path(hits[0]) if hits else None

for rel in need:
    src = find(Path(rel).name)
    if src is None:
        print("found under /kaggle/input:")
        for p in sorted(glob.glob("/kaggle/input/**", recursive=True))[:40]:
            print("  ", p)
        raise SystemExit(f"MISSING {Path(rel).name}: add the crashvlm-data dataset (Add Input)")
    dst = DATA / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        dst.symlink_to(src)

# images.zip: Kaggle sometimes unpacks zip files on upload, then we rebuild it
zdst = DATA / "annotation/pass_a/images.zip"
zdst.parent.mkdir(parents=True, exist_ok=True)
if not zdst.exists():
    z = [p for p in glob.glob("/kaggle/input/**/images.zip", recursive=True) if os.path.isfile(p)]
    if z:
        zdst.symlink_to(z[0])
    else:
        first = find("10028_exterior__f_0000.jpg")
        if first is None:
            raise SystemExit("images.zip (or its unpacked frames) not found in the dataset")
        frames = sorted(first.parent.glob("*.jpg"))
        print(f"images.zip was unpacked by Kaggle: re-zipping {len(frames)} frames ...")
        tmp = Path("/tmp/images.zip")
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as zf:
            for f in frames:
                zf.write(f, f.name)
        zdst.symlink_to(tmp)
with zipfile.ZipFile(zdst) as zf:
    n_frames = len(zf.namelist())
assert n_frames == 9420, f"images.zip has {n_frames} frames, expected 9420"
print(f"data OK ({n_frames} frames)")

# --- code (private repo) ---
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "/kaggle/temp/crashvlm" if os.path.isdir("/kaggle/temp") else "/tmp/crashvlm"
if not os.path.exists(REPO + "/scripts"):
    r = subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/Faheem8585/crashvlm.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed (check the token): " + r.stderr.replace(token, "***"))
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
os.chdir(REPO)
print("code ready:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

r = subprocess.run("pip -q install -e '.[ml]' bitsandbytes && pip -q install -U transformers",
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("pip install failed:\n" + r.stderr[-3000:])
# Kaggle ships torchao 0.10, which new peft rejects when LoRA is applied; it is not used here
subprocess.run("pip uninstall -y -q torchao", shell=True)
sys.path.insert(0, REPO + "/src")
import crashvlm, torch, transformers, peft  # noqa: F401
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers",
      transformers.__version__, "| GPUs", torch.cuda.device_count())

def run(cmd):
    """Run a step, print its output live, stop the notebook if it fails."""
    print(f"\n$ {cmd}   [{(time.time() - T0) / 3600:.1f} h elapsed]", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}   # show progress lines live, not at the end
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=env)
    for line in p.stdout:
        if "HTTP Request" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"step failed: {cmd}")

print("setup OK")


## 2. Smoke test (interactive only, ~10 min): 2 training steps + 1 report

In [ ]:
if BATCH:
    print("skipped in the committed run")
else:
    run("python scripts/train_lora.py vlm.model=Qwen/Qwen3.5-4B vlm.run_name=smoke "
        "sft.max_steps=2 sft.grad_accum=1 sft.val_limit=1")
    run(f"python scripts/run_vlm.py vlm.model=Qwen/Qwen3.5-4B vlm.run_name=smoke vlm.limit=1 "
        f"vlm.adapter={DATA}/lora/smoke/adapter")
    import json
    rec = json.loads(open(DATA / "vlm/smoke.jsonl").readline())
    print("parse_ok:", rec["parse_ok"], "| latency", round(rec["latency_s"]), "s")
    print(rec["text"][:600])
    subprocess.run(f"rm -rf {DATA}/lora/smoke {DATA}/vlm/smoke.jsonl", shell=True)
    print("\nsmoke test OK: now Save Version -> Save & Run All (Commit)")


## 3. D: LoRA fine-tuning of Qwen3.5-4B (committed run only)

Training stops after at most 5.5 h (the best adapter so far is kept), leaving time for the
evaluation below within Kaggle's 12 h limit.

In [ ]:
if BATCH:
    run("python scripts/train_lora.py vlm.model=Qwen/Qwen3.5-4B vlm.run_name=qwen3.5-4b_lora "
        "sft.time_budget_h=5.5")
else:
    print("skipped in interactive mode")


## 4. D: evaluate the adapter on the 46 test clips (~1.5 h)

In [ ]:
if BATCH:
    run(f"python scripts/run_vlm.py vlm.model=Qwen/Qwen3.5-4B vlm.run_name=qwen3.5-4b_lora "
        f"vlm.adapter={DATA}/lora/qwen3.5-4b_lora/adapter")
else:
    print("skipped in interactive mode")


## 5. C: Qwen3-VL-8B zero-shot in 4-bit (only if enough time is left)

In [ ]:
if BATCH:
    left = 11.5 - (time.time() - T0) / 3600
    if left > 2.0:
        try:   # optional: a failure here must not lose the LoRA results
            run("python scripts/run_vlm.py vlm.model=Qwen/Qwen3-VL-8B-Instruct "
                "vlm.load_in_4bit=true vlm.run_name=qwen3-vl-8b-4bit_zs")
        except RuntimeError as e:
            print("step C failed, continuing:", e)
    else:
        print(f"skipping step C: only {left:.1f} h left")
else:
    print("skipped in interactive mode")


## 6. Pack the results into one file: `crashvlm_gpu_results.zip`

In [ ]:
import shutil
out = Path("/kaggle/working/crashvlm_gpu_results.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in list(DATA.glob("vlm/*.jsonl")) + list(DATA.glob("lora/**/*")):
        if p.is_file():
            zf.write(p, p.relative_to(DATA))
            print("  +", p.relative_to(DATA))
shutil.rmtree(DATA)   # the input links and copies are not needed in the output
print(f"-> {out} ({out.stat().st_size / 2**20:.0f} MB), total {(time.time() - T0) / 3600:.1f} h")
